# Notebook 07 - GARCH Conditional Volatility

The earlier high-low range is not GARCH volatility. GARCH models volatility clustering in log returns. For $r_t=\mu+\epsilon_t$, variance is $\sigma_t^2=\omega+\alpha\epsilon_{t-1}^2+\beta\sigma_{t-1}^2$. Here $t$ is a trading day, $\mu$ is the mean, $\epsilon_t$ is a shock, $\sigma_t$ is volatility, $\omega$ is the intercept, and $\alpha+\beta$ is persistence. Returns are multiplied by 100 for estimation; pct volatility is divided by 100 for decimal units. Student-t is the pre-specified main model; normal innovations are a robustness comparison. Full-sample volatility is retrospective only and must not be used predictively.

The training Student-t persistence is near-unit: alpha + beta is below 1, but shocks are estimated to fade very slowly. This does not invalidate the conditional-volatility series, but unconditional variance and long-horizon interpretation are sensitive; the limitation must be retained in the final paper.

In [1]:
from pathlib import Path
import sys, pandas as pd
PROJECT_ROOT=Path.cwd().resolve().parent; sys.path.insert(0,str(PROJECT_ROOT/'src'))
from market_regime.config import load_research_config
from market_regime.garch import build_garch_features, save_outputs
config=load_research_config(PROJECT_ROOT/'configs'/'research_config.yaml')
features=pd.read_csv(PROJECT_ROOT/'data'/'processed'/'market_features_pre_garch.csv',index_col='Date',parse_dates=True)


In [2]:
post_garch, results=build_garch_features(features,config['data']['training_end'])
post_garch.to_csv(PROJECT_ROOT/'data'/'processed'/'market_features_with_garch.csv',index_label='Date')
save_outputs(results,PROJECT_ROOT/'outputs')
display(pd.DataFrame([{'model':k,**results[k]} for k in ('student','normal','full')]))
display(pd.DataFrame(results['diagnostics']))
display(post_garch[['GARCH_Volatility_TrainFit','GARCH_Volatility_FullSample','VIX_Close']].describe())
print('Train-fit test volatility uses fixed training parameters and prior-day shocks only. Regimes have not been estimated.')

print('Half-lives (trading days):', results['student']['shock_half_life_days'], results['full']['shock_half_life_days'])
print('ARCH-LM and Ljung-Box diagnostics:')
display(pd.DataFrame(results['diagnostics']))


,model,converged,convergence_flag,optimizer_message,nobs,loglikelihood,aic,bic,mu,omega,alpha[1],beta[1],alpha_plus_beta,shock_half_life_days,nu
0,student,True,0,Optimization terminated successfully,4527,-6098.256166,12206.512332,12238.601406,0.075475,0.009369,0.106532,0.892462,0.998995,689.200214,6.341358
1,normal,True,0,Optimization terminated successfully,4527,-6193.804364,12395.608728,12421.279987,0.059241,0.017058,0.101206,0.884999,0.986205,49.898453,NaN
2,full,True,0,Optimization terminated successfully,6640,-8951.161714,17912.323427,17946.327764,0.090167,0.016127,0.129369,0.866330,0.995699,160.823398,6.078822


,model,resid_mean,resid_std,skewness,excess_kurtosis,lb_resid_stat_10,lb_resid_p_10,lb_sq_resid_stat_10,lb_sq_resid_p_10,lb_resid_stat_20,lb_resid_p_20,lb_sq_resid_stat_20,lb_sq_resid_p_20,arch_lm_lags,arch_lm_stat,arch_lm_pvalue
0,training_student_t,-0.063048,0.995436,-0.528813,2.236855,22.608236,0.012289,12.656228,0.243532,34.750547,0.021474,24.607505,0.216861,10,12.520548,0.251727
1,training_normal,-0.043719,0.999515,-0.469681,1.773909,21.737838,0.016497,18.006654,0.054851,34.731859,0.021580,24.847511,0.207327,10,17.814836,0.058169
2,full_student_t,-0.070237,0.995495,-0.618773,2.237558,18.749423,0.043563,13.972810,0.174235,28.937848,0.088990,23.655065,0.257784,10,13.989616,0.173466


,GARCH_Volatility_TrainFit,GARCH_Volatility_FullSample,VIX_Close
count,6640.000000,6640.000000,6641.000000
mean,0.010733,0.010748,19.836071
std,0.006402,0.006378,8.316316
min,0.003579,0.003886,9.140000
25%,0.006783,0.006815,14.030000
50%,0.009032,0.009058,17.820000
75%,0.012701,0.012652,23.209999
max,0.063839,0.068227,82.690002


Train-fit test volatility uses fixed training parameters and prior-day shocks only. Regimes have not been estimated.
Half-lives (trading days): 689.2002138209175 160.82339792904716
ARCH-LM and Ljung-Box diagnostics:


,model,resid_mean,resid_std,skewness,excess_kurtosis,lb_resid_stat_10,lb_resid_p_10,lb_sq_resid_stat_10,lb_sq_resid_p_10,lb_resid_stat_20,lb_resid_p_20,lb_sq_resid_stat_20,lb_sq_resid_p_20,arch_lm_lags,arch_lm_stat,arch_lm_pvalue
0,training_student_t,-0.063048,0.995436,-0.528813,2.236855,22.608236,0.012289,12.656228,0.243532,34.750547,0.021474,24.607505,0.216861,10,12.520548,0.251727
1,training_normal,-0.043719,0.999515,-0.469681,1.773909,21.737838,0.016497,18.006654,0.054851,34.731859,0.021580,24.847511,0.207327,10,17.814836,0.058169
2,full_student_t,-0.070237,0.995495,-0.618773,2.237558,18.749423,0.043563,13.972810,0.174235,28.937848,0.088990,23.655065,0.257784,10,13.989616,0.173466
